# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructField, StringType, IntegerType, FloatType, DoubleType, DateType, StructType
from pyspark.sql.window import Window
from pyspark.sql.session import SparkSession
import warnings
warnings.filterwarnings("ignore")

In [0]:
list_months = sorted(
                        set(
                               row.tableName.split('_')[2]
                               for row in spark.sql("SHOW TABLES IN bronze").collect()
                               if row.tableName.startswith('medicoes_')
                           )
                    )

Creating a view joining all silver tables and selecting the most importante fields to carry to dashboard

In [0]:
# Joining first the dimensions table:

df_dim = (
           spark.table('silver.contracts_uc')
                .alias('c')
                ##################################################################
                #                            CUSTOMERS
                .join(
                       spark.table('silver.customers_crm')
                            .select(
                                     'customer_id',
                                     'segment',
                                     'onboarding_date',
                                   )
                            .alias('crm')
                       ,on = (F.col('c.customer_id') == F.col('crm.customer_id'))
                       ,how = 'inner'
                     )
                ##################################################################
                #                              FEES
                .join(
                       spark.table('silver.fees')
                            .select(
                                     'uf',
                                     'te_real_by_kwh',
                                     'tusd_real_by_kwh',
                                     'tusd_wire_b_real_by_kwh',
                                     'icms_tax',
                                     'pis_cofins_tax',
                                     F.col('start_date').alias('start_date_fee'),
                                     F.col('end_date').alias('end_date_fee'),
                                   )
                            .alias('f')
                       ,on = (F.col('c.uf') == F.col('f.uf'))
                       ,how = 'inner'
                     )
                ##################################################################
                #                              PLANTS
                .join(
                       spark.table('silver.plants')
                            .withColumn(
                                         'plant_type',
                                         F.when(
                                                  F.col('date_access_protocol') < '2023-01-07',
                                                  F.lit('GD I')
                                               ).otherwise(F.lit('GD II'))
                                       )
                            .select(
                                     'plant_id',
                                     'plant_name',
                                     'distributor',
                                     'power_kwp',
                                     'plant_type',
                                   )
                            .alias('p')
                       ,on = (F.col('p.plant_id') == F.col('c.plant_id'))
                       ,how = 'inner'
                     )
                ##################################################################
                .select(
                         'c.uc',
                         'c.contract_id',
                         F.col('c.start_date').alias('contract_start_date'),
                         F.col('c.end_date').alias('contract_end_date'),
                         'c.customer_id',
                         'c.discount_pct',
                         'c.distributor',
                         'c.plant_id',
                         'c.uf',
                         'c.month_limit_kwh',
                         'f.te_real_by_kwh',
                         'f.tusd_real_by_kwh',
                         'f.tusd_wire_b_real_by_kwh',
                         'f.icms_tax',
                         'f.pis_cofins_tax',
                         'f.start_date_fee',
                         'f.end_date_fee',
                         'p.plant_name',
                         'p.power_kwp',
                         'p.plant_type',
                       ) 
         )

The "df_dim" will present some duplications because an uf can change their fees through the period

In [0]:
df_total = None
for m in list_months:
    df = (
            spark.table(f'silver.measurements_{m}')
                 .alias('m')
                 # exclude rows flagged in quarantine for this month (matched by invoice_id)
                 .join(
                        spark.table(f'quarantine.warning_measurements_{m}')
                             .select('invoice_id', 'motive')
                             .alias('q')
                       ,on = (F.col('m.invoice_id') == F.col('q.invoice_id'))
                       ,how = 'left'
                      )
                 .join(
                        df_dim.alias('d')
                        ,on = (
                                  (F.col('m.uc') == F.col('d.uc'))
                                & (F.col('m.reading_at') >= F.col('d.start_date_fee'))
                                & ((F.col('m.reading_at') < F.col('d.end_date_fee')) | (F.col('d.end_date_fee').isNull()))
                              )
                        ,how = 'inner'
                      )
                 .join(
                        spark.table('silver.tax_flags')
                             .alias('t')
                        ,on = (F.lit(m[:4] + '-' + m[4:]) == F.col('t.month'))
                        ,how = 'inner'
                      )
                 .join(
                        spark.table('silver.law_14300_parameters')
                             .alias('l')
                        ,on = (F.lit(m[:4]) == F.col('l.year_start_contract'))
                        ,how = 'inner'
                      )
                 .withColumn('reference_date', F.lit(m))
                 .withColumn('year', F.left(F.col('reference_date'), F.lit(4)))
                 .withColumn('month', F.right(F.col('reference_date'), F.lit(2)))
                 .withColumn(
                              'contract_status',
                              F.when(
                                      (F.col('reading_at') >= F.col('contract_start_date'))
                                    & (
                                        F.col('contract_end_date').isNull()
                                      | (F.col('reading_at') <= F.col('contract_end_date'))
                                      ),
                                      F.lit('ACTIVE')
                                    ).otherwise(F.lit('INACTIVE'))
                            )
                 .withColumn(
                              'in_quarantine',
                              F.when(F.col('q.invoice_id').isNotNull(), F.lit(True))
                                .otherwise(F.lit(False))
                            )
                 .select(
                          'reference_date',
                          'year',
                          'month',
                          'm.reading_at',
                          'm.regional',
                          'm.uc',
                          'm.invoice_id',
                          'm.consume_kwh',
                          'm.compensated_energy_kwh',
                          'm.balance_kwh',
                          'contract_status',
                          *[F.col(f'd.{c}') for c in df_dim.columns if c != 'uc'],
                          't.flag',
                          't.additional_charge_kwh',
                          'l.pct_b_wire_not_compensated',
                          'in_quarantine',
                          'motive',
                          'm._ingestion_ts'
                        )
         )
    df_total = df if df_total is None else df_total.union(df)
    

# Billing Calculations

Let's create the auxiliary calculations and the total invoiced by the company:

In [0]:
df_total = (
             df_total.withColumn(
                                  'full_charge_applied'
                                  ,F.when(
                                            (F.col('in_quarantine') == False) 
                                          & (F.col('contract_status') == 'ACTIVE')
                                          ,(F.col('te_real_by_kwh') + F.col('tusd_real_by_kwh') + F.col('additional_charge_kwh'))
                                           /
                                           (F.lit(1) - F.col('icms_tax') - F.col('pis_cofins_tax'))
                                         ).otherwise(F.lit(0))
                                )
                     .withColumn(
                                  'wire_b_deducted'
                                  ,F.when(
                                              (F.col('plant_type') == 'GD II')
                                            & (F.col('in_quarantine') == False)
                                            & (F.col('contract_status') == 'ACTIVE')
                                            ,(F.col('compensated_energy_kwh') * F.col('tusd_wire_b_real_by_kwh') * F.col('pct_b_wire_not_compensated'))
                                             /
                                             (F.lit(1) - F.col('icms_tax') - F.col('pis_cofins_tax'))
                                         ).otherwise(F.lit(0))
                                )
                     .withColumn(
                                  'invoiced_value'
                                  ,F.when(
                                            (F.col('in_quarantine') == False)
                                          & (F.col('contract_status') == 'ACTIVE')
                                          ,F.round(
                                                    (
                                                        F.col('compensated_energy_kwh') 
                                                      * F.col('full_charge_applied') 
                                                      * (F.lit(1) - F.col('discount_pct'))
                                                    )
                                                    -
                                                    (F.col('wire_b_deducted'))
                                                    ,2
                                                  )
                                         ).otherwise(F.lit(0))
                                )
                     .withColumn('invoice_id_NEW', F.col('invoice_id'))
                     .withColumn('kwh_invoiced_NEW', F.col('compensated_energy_kwh'))
                     .withColumn('total_fees_correctly_NEW', F.col('full_charge_applied'))
                     .withColumn('discount_pct_applied_NEW', F.col('discount_pct'))
                     .withColumn('wire_b_cost_correctly_NEW', F.col('wire_b_deducted'))
                     .withColumn('total_invoiced_NEW', F.col('invoiced_value'))
                     .drop('invoice_id', 'compensated_energy_kwh', 'full_charge_applied', 'discount_pct', 'wire_b_deducted', 'invoiced_value')
           )

In [0]:
df_total = (
             df_total.alias('p')
                     .join(
                            spark.table('silver.invoice_erp_202510_to_202609')
                                 .withColumn(
                                              'rn'
                                             , F.row_number().over(Window.partitionBy('uc','month').orderBy(F.desc('invoice_date')))
                                            )
                                 .filter(F.col('rn') == F.lit(1))
                                 .drop('rn')
                                 .select(
                                          'invoice_id',
                                          'uc',
                                          'month',
                                          'kwh_invoiced',
                                          'full_charge_applied',
                                          'discount_applied',
                                          'wire_b_deducted',
                                          'invoiced_value',
                                        )
                                 .alias('aux')
                            ,on = (
                                      (F.col('p.uc') == F.col('aux.uc'))
                                    & (F.col('p.reference_date') == F.replace(F.col('aux.month'), F.lit('-'), F.lit('')))
                                  )
                            ,how = 'left'
                          )
                     .drop(F.col('aux.uc'),F.col('aux.month'))
                     .withColumnsRenamed({
                                            'invoice_id' : 'invoice_id_OLD',
                                            'kwh_invoiced' : 'kwh_invoiced_OLD',
                                            'full_charge_applied' : 'total_fees_correctly_OLD',
                                            'discount_applied' : 'discount_pct_applied_OLD',
                                            'wire_b_deducted' : 'wire_b_cost_correctly_OLD',
                                            'invoiced_value' : 'total_invoiced_OLD'
                                        })
                     # improper only when the ERP actually issued an invoice for the row
                     .withColumn(
                                  'improper_invoiced',
                                  F.when(
                                          (
                                            (F.col('in_quarantine') == True)
                                          | (F.col('contract_status') == 'INACTIVE')
                                          )
                                        & F.col('invoice_id_OLD').isNotNull()
                                        , F.lit(True)
                                        ).otherwise(F.lit(False))
                                )
           )

### Contractual Compliance & Savings Metrics

In [0]:
df_total = (
             df_total.withColumns(
                                  {
                                    # flag: ERP invoice exceeded the contractual monthly kWh limit
                                    'flag_invoiced_higher_contract': F.when(
                                                                               F.col('kwh_invoiced_OLD').isNotNull()
                                                                               & (F.col('kwh_invoiced_OLD') > F.col('month_limit_kwh')),
                                                                               F.lit(True)
                                                                           ).otherwise(F.lit(False)),
                                  }
                                 )
                     .withColumns(
                                  {
                                    # refund: full invoice when improper (quarantine or inactive); proportional excess when above limit
                                    'total_refund_to_customer': F.when(
                                                                        F.col('improper_invoiced') == True,
                                                                        F.round(F.col('total_invoiced_OLD'), 2)
                                                                      )
                                                                 .when(
                                                                        F.col('flag_invoiced_higher_contract') == True,
                                                                        F.round(
                                                                                  F.col('total_invoiced_OLD')
                                                                                * (F.col('kwh_invoiced_OLD') - F.col('month_limit_kwh'))
                                                                                / F.col('kwh_invoiced_OLD'),
                                                                                2
                                                                               )
                                                                      ).otherwise(F.lit(0)),
                                    # savings: discount applied × reference value (kwh × full charge before discount)
                                    'total_BRL_saved': F.when(
                                                                (F.col('in_quarantine') == False)
                                                                & (F.col('contract_status') == 'ACTIVE'),
                                                                F.round(
                                                                           F.col('kwh_invoiced_NEW')
                                                                           * F.col('total_fees_correctly_NEW')
                                                                           * F.col('discount_pct_applied_NEW'),
                                                                           2
                                                                       )
                                                             ).otherwise(F.lit(0)),
                                  }
                                 )
           )

In [0]:
# Order columns alphabetically
sorted_cols = sorted(df_total.columns)
df_total = df_total.select(*sorted_cols)

In [0]:
# Incremental load: never replaces gold.fact_measurements_invoiced — appends only new facts
target = 'gold.fact_measurements_invoiced'
key_cols = ['uc', 'reference_date', 'invoice_id_NEW', 'contract_id', 'start_date_fee']

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df_total.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"✓ {target} created ({df_total.count()} rows)")
else:
    # append only records whose key is not present in gold yet
    df_new = df_total.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"✓ {target} appended {n_new} new row(s)")
    else:
        print(f"↷ {target}: no new records — nothing appended")

In [0]:
spark.table('gold.fact_measurements_invoiced').show(truncate=False)